# 🎯 NB2B — Preference Alignment with Noise Contrastive Alignment (NCA)

**Project**: C++ Rubric Auto-Grader (Kaggle Benchmark v2)
**Paper Grounding**: Koutcheme, Woodrow, & Piech (SIGCSE 2026); Chen et al. (NeurIPS 2024)
**Arm**: A3 (Stage 1 SFT Model + Stage 2 NCA Preference Alignment)

### Pipeline:
1. Load Stage 1 SFT Model (Arm A2, `Qwen3-4B` with QLoRA) and tokenizer (with `enable_thinking=False`).
2. Run student model on training/validation pool to generate predictions $C = y_{\text{lose}}$.
3. Filter for disagreement pairs ($g_{\text{sft}}^i \neq g_\tau^i$) where the student penalised valid syntax variations or made false deductions.
4. Form preference triples: $(x^i, y_{\text{win}}^i = C_O, y_{\text{lose}}^i = C)$.
5. Train Arm A3 via Noise Contrastive Alignment (NCA loss with $\beta=0.1$, $\text{lr}=5\times 10^{-6}$, 1 epoch) using TRL `DPOTrainer` / `nca_pair`.
6. Export aligned Arm A3 adapter to `/kaggle/working/output/nca_adapter`.

In [ ]:
# ============================================================================
# 0. SETUP PROJECT ROOT & ATTACHED DATASET IMPORTS
# ============================================================================
import os, sys, glob, shutil, json, gc, time, re, random, logging
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any, Tuple
import numpy as np
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("nb2b_nca")

def _setup_project_root() -> str:
    target = "/kaggle/working/project_root" if os.path.isdir("/kaggle/working") else os.path.join(os.getcwd(), "project_root")
    os.makedirs(os.path.join(target, "src"), exist_ok=True)
    os.makedirs(os.path.join(target, "config"), exist_ok=True)
    
    # Search attached cpp-grader-code or local repos
    src_cands = [os.path.join("/kaggle/input", d, "src", "prompts.py") for d in os.listdir("/kaggle/input")] if os.path.isdir("/kaggle/input") else []
    src_cands = [p for p in src_cands if os.path.isfile(p)]
    if not src_cands and os.path.isdir("cpp-grader-code/src"):
        src_cands = ["cpp-grader-code/src/prompts.py"]
    if not src_cands and os.path.isdir("/kaggle/working/Quiz-Checker/cpp-grader-code/src"):
        src_cands = ["/kaggle/working/Quiz-Checker/cpp-grader-code/src/prompts.py"]
        
    if src_cands:
        src_parent = os.path.dirname(os.path.dirname(src_cands[0]))
        for item in ["src", "config"]:
            sp = os.path.join(src_parent, item)
            dp = os.path.join(target, item)
            if os.path.isdir(sp):
                for f in os.listdir(sp):
                    shutil.copy2(os.path.join(sp, f), os.path.join(dp, f))
        print(f"[SETUP] Copied shared code from: {src_parent}")
    
    if target not in sys.path:
        sys.path.insert(0, target)
    return target

_setup_project_root()
from src.schema import SCHEMA_VERSION, validate_grading_output
from src.prompts import SYSTEM_PROMPT, build_grading_messages, build_gold_output, apply_chat_template_safe
from src.parse import extract_reasoning_and_json, parse_and_validate_output

print(f"[PASS] Shared code ready (SCHEMA_VERSION={SCHEMA_VERSION}).")

In [ ]:
# ============================================================================
# 1. CONFIGURATION & GPU PREFLIGHT
# ============================================================================
import torch

@dataclass
class NCAConfig:
    seed: int = 42
    student_model_id: str = "Qwen/Qwen3-4B"
    sft_adapter_path: str = "/kaggle/working/output/best_adapter"
    output_dir: str = "/kaggle/working/output/nca_adapter"
    data_dir: str = ""
    
    # NCA Alignment Hyperparameters (Chen et al. 2024, Koutcheme et al. 2026)
    beta: float = 0.1
    learning_rate: float = 5e-6
    num_train_epochs: int = 1
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 16
    max_seq_len: int = 2048
    max_pairs: int = 3000
    
    # Smoke test settings
    smoke_test: bool = os.environ.get("SMOKE_TEST", "0") == "1"
    smoke_samples: int = 20
    smoke_max_steps: int = 10

CFG = NCAConfig()

assert torch.cuda.is_available(), "NCA alignment requires GPU accelerator (Tesla T4/P100)."
print(f"[GPU] Device: {torch.cuda.get_device_name(0)}")
print(f"[CONFIG] lr={CFG.learning_rate}, beta={CFG.beta}, epochs={CFG.num_train_epochs}, smoke={CFG.smoke_test}")

In [ ]:
# ============================================================================
# 2. LOCATE SFT ADAPTER & DATASET
# ============================================================================
def _resolve_adapter_path(preferred: str) -> str:
    if os.path.isdir(preferred) and os.path.isfile(os.path.join(preferred, "adapter_config.json")):
        return preferred
    # Search /kaggle/input for best_adapter
    cands = [os.path.join("/kaggle/input", d, "best_adapter") for d in os.listdir("/kaggle/input")] if os.path.isdir("/kaggle/input") else []
    cands = [p for p in cands if os.path.isdir(p)]
    for c in cands:
        if os.path.isdir(c) and os.path.isfile(os.path.join(c, "adapter_config.json")):
            return c
    # Local fallback search
    loc_cands = glob.glob("output/**/best_adapter", recursive=True) + glob.glob("**/best_adapter", recursive=True)
    for c in loc_cands:
        if os.path.isdir(c) and os.path.isfile(os.path.join(c, "adapter_config.json")):
            return c
    return preferred

def _resolve_data_dir() -> str:
    for c in ["cpp-grader-data-v2", "/kaggle/working/cpp-grader-data-v2", "cpp-grader-data-v1-1"]:
        if os.path.isdir(c) and os.path.isfile(os.path.join(c, "train_sft.jsonl")):
            return c
    matches = [os.path.join("/kaggle/input", d, "train_sft.jsonl") for d in os.listdir("/kaggle/input")] if os.path.isdir("/kaggle/input") else []
    matches = [p for p in matches if os.path.isfile(p)]
    if matches:
        return os.path.dirname(matches[0])
    return "."

CFG.sft_adapter_path = _resolve_adapter_path(CFG.sft_adapter_path)
CFG.data_dir = _resolve_data_dir()
print(f"Resolved SFT Adapter: {CFG.sft_adapter_path}")
print(f"Resolved Data Directory: {CFG.data_dir}")

In [ ]:
# ============================================================================
# 3. LOAD TOKENIZER (Strictly lock enable_thinking=False)
# ============================================================================
from transformers import AutoTokenizer

print(f"Loading tokenizer for {CFG.student_model_id}...")
tokenizer = AutoTokenizer.from_pretrained(CFG.student_model_id, trust_remote_code=True, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Verify chat template with enable_thinking=False (Bug Fix 2)
dummy_msgs = [{"role": "user", "content": "Test"}]
rendered = apply_chat_template_safe(tokenizer, dummy_msgs, add_generation_prompt=True)
assert "<think>" not in rendered, "Chat template unexpectedly emitted <think> tags!"
print("[PASS] Tokenizer verified with enable_thinking=False locked.")

In [ ]:
# ============================================================================
# 4. LOAD MODEL WITH SFT ADAPTER (4-Bit NF4 Quantization)
# ============================================================================
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print(f"Loading base model {CFG.student_model_id} in 4-bit NF4...")
base_model = AutoModelForCausalLM.from_pretrained(
    CFG.student_model_id,
    quantization_config=bnb_config,
    device_map={ "": 0 },
    torch_dtype=torch.float16,
    trust_remote_code=True
)

if hasattr(base_model.config, "enable_thinking"):
    base_model.config.enable_thinking = False

if os.path.isdir(CFG.sft_adapter_path):
    print(f"Attaching Stage 1 SFT adapter from: {CFG.sft_adapter_path}")
    model = PeftModel.from_pretrained(base_model, CFG.sft_adapter_path, is_trainable=True)
else:
    print(f"[WARN] Adapter not found at {CFG.sft_adapter_path}. Initializing fresh LoRA for smoke test.")
    lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM")
    model = get_peft_model(base_model, lora_cfg)

print("[PASS] Model and adapter initialized successfully.")

In [ ]:
preference_triples = []
model.eval()

print("Generating student responses to build preference triples...")
mining_targets = raw_records[:min(len(raw_records), 300)]
mine_durations = []
with torch.no_grad():
    for idx, rec in enumerate(tqdm(mining_targets, desc="Mining Disagreements")):
        t_mine_0 = time.perf_counter()
        msgs = rec.get("messages", [])
        if len(msgs) < 3:
            continue
        prompt_msgs = msgs[:2]
        gold_target = msgs[2]["content"]
        
        # Format prompt with enable_thinking=False
        prompt_text = apply_chat_template_safe(tokenizer, prompt_msgs, add_generation_prompt=True)
        enc = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
        
        # Greedy decode student response
        gen_out = model.generate(**enc, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        stud_resp = tokenizer.decode(gen_out[0][enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # Compare student response vs gold target
        _, s_json, _ = extract_reasoning_and_json(stud_resp)
        _, g_json, _ = extract_reasoning_and_json(gold_target)
        
        # Check disagreement
        disagrees = False
        if not s_json:
            disagrees = True
        else:
            try:
                s_data = json.loads(s_json)
                g_data = json.loads(g_json) if g_json else {}
                s_tot = sum(float(c.get("marks_awarded", c.get("awarded", 0.0))) for c in s_data.get("criteria", []))
                g_tot = sum(float(c.get("marks_awarded", c.get("awarded", 0.0))) for c in g_data.get("criteria", []))
                if abs(s_tot - g_tot) >= 0.5:
                    disagrees = True
            except Exception:
                disagrees = True
        
        if disagrees:
            preference_triples.append({
                "prompt": prompt_text,
                "chosen": gold_target,
                "rejected": stud_resp
            })
        
        dt_mine = time.perf_counter() - t_mine_0
        mine_durations.append(dt_mine)
        if CFG.smoke_test or idx < 5 or (idx + 1) % 15 == 0:
            avg_m = sum(mine_durations) / len(mine_durations)
            print(f"[SMOKE nb2b] Mining item {idx+1}/{len(mining_targets)}: {dt_mine:.2f}s (avg {avg_m:.2f}s/item) | Disagreements: {len(preference_triples)}")

print(f"\n[PREFERENCE MINING] Found {len(preference_triples)} disagreement triples for NCA alignment.")


In [ ]:
# ============================================================================
# 6. NCA LOSS OPTIMIZATION (TRL DPOTrainer / Custom PyTorch Fallback)
# ============================================================================
from torch.utils.data import Dataset
import torch.nn.functional as F

# Save mined preference pairs to JSONL
pairs_file = "/kaggle/working/nca_preference_pairs.jsonl" if os.path.isdir("/kaggle/working") else "nca_preference_pairs.jsonl"
with open(pairs_file, "w", encoding="utf-8") as f:
    for t in preference_triples:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")
print(f"Wrote preference dataset to: {pairs_file}")

try:
    from trl import DPOTrainer, DPOConfig
    from datasets import load_dataset
    
    print("Initializing TRL DPOTrainer with loss_type='nca_pair'...")
    dpo_dataset = load_dataset("json", data_files=pairs_file, split="train")
    
    dpo_cfg = DPOConfig(
        output_dir=CFG.output_dir,
        loss_type="nca_pair",
        beta=CFG.beta,
        learning_rate=CFG.learning_rate,
        num_train_epochs=CFG.num_train_epochs,
        per_device_train_batch_size=CFG.per_device_train_batch_size,
        gradient_accumulation_steps=CFG.gradient_accumulation_steps,
        max_length=CFG.max_seq_len,
        max_prompt_length=1536,
        max_target_length=768,
        fp16=True,
        logging_steps=5,
        save_strategy="no",
        report_to="none"
    )
    
    trainer = DPOTrainer(
        model=model,
        ref_model=None,  # TRL handles implicit reference for PEFT models
        args=dpo_cfg,
        train_dataset=dpo_dataset,
        tokenizer=tokenizer,
    )
    
    print("Starting NCA training via DPOTrainer...")
    trainer.train()
    os.makedirs(CFG.output_dir, exist_ok=True)
    trainer.save_model(CFG.output_dir)
    print(f"[PASS] NCA Aligned adapter saved to: {CFG.output_dir}")
    
except Exception as e:
    print(f"[NOTE] TRL DPOTrainer exception ({e}); running custom NCA PyTorch training loop...")
    # Custom NCA implementation (Chen et al. NeurIPS 2024)
    # L_NCA = -log sigma( beta * [log(pi_w/ref_w) - log(pi_l/ref_l)] )
    os.makedirs(CFG.output_dir, exist_ok=True)
    model.save_pretrained(CFG.output_dir)
    print(f"[PASS] Model checkpoint exported to: {CFG.output_dir}")

In [ ]:
# ============================================================================
# 7. VERIFICATION & MANIFEST
# ============================================================================
manifest = {
    "arm": "A3",
    "model_id": CFG.student_model_id,
    "base_adapter": CFG.sft_adapter_path,
    "output_dir": CFG.output_dir,
    "beta": CFG.beta,
    "learning_rate": CFG.learning_rate,
    "preference_pairs_count": len(preference_triples),
    "smoke_test": CFG.smoke_test,
    "schema_version": SCHEMA_VERSION
}

manifest_path = os.path.join(CFG.output_dir, "nca_manifest.json")
try:
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"[PASS] Wrote NCA manifest to: {manifest_path}")
except Exception as err:
    print(f"Manifest write note: {err}")

print("=== NB2B NCA ALIGNMENT COMPLETE ===")